# Multimodal Late Fusion: EEG Feature Comparison

This notebook compares different EEG feature extraction approaches **in the context of full multimodal late fusion**.

For each EEG feature set, we run complete late fusion with:
- **Physiology** (pupil metrics)
- **Behavior** (reaction time, decision time, etc.)
- **Gaze** (gaze position, movements, fixations)
- **EEG** (different feature extraction approaches)

## EEG Feature Sets Tested:
1. **Regional** (16 features: 4 bands × 4 regions)
2. **Regional + Channels** (96 features: 16 regional + 80 individual channels)
3. **Non-temporal** (96 features: power + temporal dynamics + lateralization)

## Key Questions:
1. Which EEG feature approach gives best **overall fusion accuracy**?
2. Which EEG approach has highest **modality contribution** (weight in weighted fusion)?
3. Which specific EEG features are most important for decision prediction?
4. Does adding channel-level or temporal information improve beyond regional averages?

In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================
TIMEFRAME = 'PRE'  # Fixed to PRE - EEG is from display/decision phase
# ============================================================================

import sys
sys.path.append('../..')

import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.utils.io import load_features, save_results
from src.utils.config import get_model_params
from src.utils.validation import validate_modality_features
from src.models.fusion import weighted_late_fusion
from src.visualization.plots import set_style, plot_method_comparison, plot_modality_weights

np.random.seed(42)
set_style('whitegrid')

print(f"\n{'='*70}")
print(f"MULTIMODAL LATE FUSION: EEG FEATURE COMPARISON ({TIMEFRAME})")
print(f"{'='*70}\n")


MULTIMODAL LATE FUSION: EEG FEATURE COMPARISON (PRE)



## 1. Load Base Features (Physio, Behavior, Gaze)

In [2]:
# Load pre-extracted features
features_path = f'../../data/results/features_{TIMEFRAME}/extracted_features_{TIMEFRAME}.pkl'
feature_data = load_features(features_path, timeframe=TIMEFRAME)

merged_df = feature_data['merged_df']
physio_cols = feature_data['physio_cols']
behavior_cols = feature_data['behavior_cols']
gaze_cols = feature_data['gaze_cols']

print(f"Loaded {len(merged_df)} trials from {merged_df['subject_id'].nunique()} subjects")
print(f"\nFeature counts:")
print(f"  Physiology: {len(physio_cols)} features")
print(f"  Behavior:   {len(behavior_cols)} features")
print(f"  Gaze:       {len(gaze_cols)} features")

ERROR: Feature file not found: ../../data/results/features_PRE/extracted_features_PRE.pkl
  → Run notebooks/preprocessing/feature_extraction_PRE.ipynb first


FileNotFoundError: [Errno 2] No such file or directory: '../../data/results/features_PRE/extracted_features_PRE.pkl'

## 2. Load All EEG Feature Sets

In [ ]:
# Load all three EEG feature sets
eeg_datasets = {}

# 1. Regional only
try:
    with open('../../data/results/eeg_features.pkl', 'rb') as f:
        data = pickle.load(f)
        eeg_datasets['Regional'] = {
            'df': data['eeg_features_df'],
            'cols': data['feature_columns']
        }
    print(f"✓ Regional: {len(eeg_datasets['Regional']['cols'])} features")
except FileNotFoundError:
    print("✗ Regional features not found - run extract_eeg_features.py")

# 2. Regional + Channels
try:
    with open('../../data/results/eeg_features_with_channels.pkl', 'rb') as f:
        data = pickle.load(f)
        eeg_datasets['Regional+Channels'] = {
            'df': data['eeg_features_df'],
            'cols': data['feature_columns']
        }
    print(f"✓ Regional+Channels: {len(eeg_datasets['Regional+Channels']['cols'])} features")
except FileNotFoundError:
    print("✗ Regional+Channels not found - run extract_eeg_features_with_channels.py")

# 3. Non-temporal
try:
    with open('../../data/results/eeg_features_non_temporal.pkl', 'rb') as f:
        data = pickle.load(f)
        eeg_datasets['Non-temporal'] = {
            'df': data['eeg_features_df'],
            'cols': data['feature_columns']
        }
    print(f"✓ Non-temporal: {len(eeg_datasets['Non-temporal']['cols'])} features")
except FileNotFoundError:
    print("✗ Non-temporal not found - run extract_eeg_features_non_temporal.py")

if len(eeg_datasets) == 0:
    raise RuntimeError("No EEG features loaded. Run extraction scripts first.")

## 3. Run Late Fusion for Each EEG Feature Set

In [ ]:
def run_late_fusion_with_eeg(merged_df, physio_cols, behavior_cols, gaze_cols,
                              eeg_df, eeg_cols, eeg_name):
    """
    Run full late fusion with physio, behavior, gaze, and specified EEG features.
    
    Returns
    -------
    dict
        Fusion results including accuracy, weights, and modality contributions
    """
    print(f"\n{'='*70}")
    print(f"LATE FUSION WITH {eeg_name} EEG FEATURES")
    print(f"{'='*70}")
    
    # Filter to EEG subjects and merge
    eeg_subjects = eeg_df['subject_id'].unique()
    merged_df_eeg = merged_df[merged_df['subject_id'].isin(eeg_subjects)].copy()
    
    merged_df_eeg = merged_df_eeg.merge(
        eeg_df[['trial_id'] + eeg_cols],
        on='trial_id',
        how='inner'
    )
    
    print(f"Subjects: {merged_df_eeg['subject_id'].nunique()}")
    print(f"Trials:   {len(merged_df_eeg)}")
    print(f"\nFeature counts:")
    print(f"  Physiology: {len(physio_cols)}")
    print(f"  Behavior:   {len(behavior_cols)}")
    print(f"  Gaze:       {len(gaze_cols)}")
    print(f"  EEG:        {len(eeg_cols)}")
    
    # Prepare modality arrays
    X_physio = SimpleImputer(strategy='mean').fit_transform(merged_df_eeg[physio_cols])
    X_behavior = SimpleImputer(strategy='mean').fit_transform(merged_df_eeg[behavior_cols])
    X_gaze = SimpleImputer(strategy='mean').fit_transform(merged_df_eeg[gaze_cols]) if len(gaze_cols) > 0 else np.zeros((len(merged_df_eeg), 1))
    X_eeg = SimpleImputer(strategy='mean').fit_transform(merged_df_eeg[eeg_cols])
    
    y = merged_df_eeg['outcome'].values
    subjects = merged_df_eeg['subject_id'].values
    
    # Validate before fusion
    modality_names = ['Physiology', 'Behavior', 'Gaze', 'EEG']
    X_modalities = [X_physio, X_behavior, X_gaze, X_eeg]
    validate_modality_features(X_modalities, y, subjects, modality_names)
    
    # Run all three fusion methods
    print("\nRunning fusion methods...")
    results_avg = weighted_late_fusion(X_modalities, y, subjects, modality_names,
                                       fusion_method='average')
    results_weighted = weighted_late_fusion(X_modalities, y, subjects, modality_names,
                                            fusion_method='weighted')
    results_stacking = weighted_late_fusion(X_modalities, y, subjects, modality_names,
                                            fusion_method='stacking')
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"RESULTS ({eeg_name})")
    print(f"{'='*70}")
    print(f"\nFusion Performance:")
    print(f"  Average:  Acc={results_avg['accuracy_mean']:.3f}, F1={results_avg['f1_mean']:.3f}")
    print(f"  Weighted: Acc={results_weighted['accuracy_mean']:.3f}, F1={results_weighted['f1_mean']:.3f}")
    print(f"  Stacking: Acc={results_stacking['accuracy_mean']:.3f}, F1={results_stacking['f1_mean']:.3f}")
    
    print(f"\nModality Weights (Weighted Fusion):")
    for name, w in zip(modality_names, results_weighted['weights']):
        print(f"  {name:12s}: {w*100:5.1f}%")
    
    return {
        'eeg_name': eeg_name,
        'n_eeg_features': len(eeg_cols),
        'n_subjects': merged_df_eeg['subject_id'].nunique(),
        'n_trials': len(merged_df_eeg),
        'results_avg': results_avg,
        'results_weighted': results_weighted,
        'results_stacking': results_stacking,
        'modality_names': modality_names
    }

In [ ]:
# Run late fusion for each EEG feature set
fusion_results = {}

for eeg_name, eeg_data in eeg_datasets.items():
    fusion_results[eeg_name] = run_late_fusion_with_eeg(
        merged_df,
        physio_cols,
        behavior_cols,
        gaze_cols,
        eeg_data['df'],
        eeg_data['cols'],
        eeg_name
    )

## 4. Compare EEG Feature Sets: Overall Performance

In [ ]:
# Build comparison dataframe
comparison_rows = []

for eeg_name, result in fusion_results.items():
    comparison_rows.extend([
        {
            'EEG Features': eeg_name,
            'N EEG Features': result['n_eeg_features'],
            'Method': 'Average',
            'Accuracy': result['results_avg']['accuracy_mean'],
            'Accuracy SEM': result['results_avg']['accuracy_sem'],
            'F1-Score': result['results_avg']['f1_mean']
        },
        {
            'EEG Features': eeg_name,
            'N EEG Features': result['n_eeg_features'],
            'Method': 'Weighted',
            'Accuracy': result['results_weighted']['accuracy_mean'],
            'Accuracy SEM': result['results_weighted']['accuracy_sem'],
            'F1-Score': result['results_weighted']['f1_mean']
        },
        {
            'EEG Features': eeg_name,
            'N EEG Features': result['n_eeg_features'],
            'Method': 'Stacking',
            'Accuracy': result['results_stacking']['accuracy_mean'],
            'Accuracy SEM': result['results_stacking']['accuracy_sem'],
            'F1-Score': result['results_stacking']['f1_mean']
        }
    ])

comparison_df = pd.DataFrame(comparison_rows)

print(f"\n{'='*80}")
print(f"COMPARISON: MULTIMODAL LATE FUSION WITH DIFFERENT EEG FEATURES ({TIMEFRAME})")
print(f"{'='*80}")
print(comparison_df.to_string(index=False))

# Find best configuration
best = comparison_df.loc[comparison_df['Accuracy'].idxmax()]
print(f"\n{'='*80}")
print(f"BEST CONFIGURATION:")
print(f"  EEG Features: {best['EEG Features']} ({int(best['N EEG Features'])} features)")
print(f"  Fusion Method: {best['Method']}")
print(f"  Accuracy: {best['Accuracy']:.3f} ± {best['Accuracy SEM']:.3f}")
print(f"  F1-Score: {best['F1-Score']:.3f}")
print(f"{'='*80}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Group by fusion method
for method_idx, method in enumerate(['Weighted', 'Stacking']):
    ax = axes[method_idx]
    method_df = comparison_df[comparison_df['Method'] == method]
    
    x_pos = np.arange(len(method_df))
    colors = sns.color_palette('Set2', len(method_df))
    
    ax.bar(x_pos, method_df['Accuracy'], yerr=method_df['Accuracy SEM'],
           color=colors, capsize=5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(method_df['EEG Features'], rotation=15, ha='right')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{method} Fusion: Accuracy by EEG Feature Set', fontweight='bold')
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
    
    # Add feature counts as text
    for i, (idx, row) in enumerate(method_df.iterrows()):
        ax.text(i, row['Accuracy'] + 0.01, f"n={int(row['N EEG Features'])}",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Compare EEG Feature Sets: Modality Contributions

In [ ]:
# Extract modality weights for weighted fusion
weights_comparison = []

for eeg_name, result in fusion_results.items():
    weights = result['results_weighted']['weights']
    modality_names = result['modality_names']
    
    for mod_name, weight in zip(modality_names, weights):
        weights_comparison.append({
            'EEG Features': eeg_name,
            'Modality': mod_name,
            'Weight': weight * 100  # Convert to percentage
        })

weights_df = pd.DataFrame(weights_comparison)

print(f"\n{'='*70}")
print(f"MODALITY CONTRIBUTIONS (Weighted Fusion)")
print(f"{'='*70}")
for eeg_name in weights_df['EEG Features'].unique():
    print(f"\n{eeg_name}:")
    subset = weights_df[weights_df['EEG Features'] == eeg_name]
    for _, row in subset.iterrows():
        print(f"  {row['Modality']:12s}: {row['Weight']:5.1f}%")

In [ ]:
# Visualize modality weights
fig, ax = plt.subplots(figsize=(12, 6))

# Pivot for grouped bar chart
weights_pivot = weights_df.pivot(index='Modality', columns='EEG Features', values='Weight')

weights_pivot.plot(kind='bar', ax=ax, width=0.8, color=sns.color_palette('Set2', len(fusion_results)))
ax.set_ylabel('Weight (%)')
ax.set_xlabel('')
ax.set_title('Modality Contributions by EEG Feature Set (Weighted Fusion)', fontweight='bold')
ax.legend(title='EEG Features', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Highlight EEG contribution
print(f"\n{'='*70}")
print(f"EEG MODALITY CONTRIBUTION")
print(f"{'='*70}")
eeg_weights = weights_df[weights_df['Modality'] == 'EEG'].sort_values('Weight', ascending=False)
print(eeg_weights[['EEG Features', 'Weight']].to_string(index=False))
print(f"\nBest EEG contribution: {eeg_weights.iloc[0]['EEG Features']} ({eeg_weights.iloc[0]['Weight']:.1f}%)")

## 6. Statistical Comparison

In [ ]:
# Compare weighted fusion accuracy across EEG feature sets
print(f"\n{'='*70}")
print(f"STATISTICAL COMPARISONS (Weighted Fusion Accuracy)")
print(f"{'='*70}\n")

eeg_names = list(fusion_results.keys())
for i in range(len(eeg_names)):
    for j in range(i+1, len(eeg_names)):
        name1, name2 = eeg_names[i], eeg_names[j]
        
        acc1 = fusion_results[name1]['results_weighted']['accuracy_mean']
        sem1 = fusion_results[name1]['results_weighted']['accuracy_sem']
        
        acc2 = fusion_results[name2]['results_weighted']['accuracy_mean']
        sem2 = fusion_results[name2]['results_weighted']['accuracy_sem']
        
        diff = acc1 - acc2
        diff_sem = np.sqrt(sem1**2 + sem2**2)
        
        print(f"{name1} vs {name2}:")
        print(f"  Difference: {diff:+.4f} ± {diff_sem:.4f}")
        
        if abs(diff) > 2 * diff_sem:
            print(f"  ✓ Significant difference (p < 0.05)")
        else:
            print(f"  No significant difference")
        print()

## 7. Summary and Recommendations

In [ ]:
print(f"\n{'='*70}")
print(f"SUMMARY: MULTIMODAL LATE FUSION WITH EEG")
print(f"{'='*70}")

# Best overall performance
weighted_results = comparison_df[comparison_df['Method'] == 'Weighted'].sort_values('Accuracy', ascending=False)
best_perf = weighted_results.iloc[0]

print(f"\n1. BEST OVERALL PERFORMANCE:")
print(f"   EEG Features: {best_perf['EEG Features']} ({int(best_perf['N EEG Features'])} features)")
print(f"   Accuracy: {best_perf['Accuracy']:.3f} ± {best_perf['Accuracy SEM']:.3f}")
print(f"   F1-Score: {best_perf['F1-Score']:.3f}")

# EEG contribution
eeg_weights_sorted = weights_df[weights_df['Modality'] == 'EEG'].sort_values('Weight', ascending=False)
best_eeg_contrib = eeg_weights_sorted.iloc[0]

print(f"\n2. HIGHEST EEG CONTRIBUTION:")
print(f"   EEG Features: {best_eeg_contrib['EEG Features']}")
print(f"   Weight: {best_eeg_contrib['Weight']:.1f}%")

# Comparison to baseline
print(f"\n3. COMPARISON TO BASELINE (Regional EEG):")
if 'Regional' in fusion_results:
    baseline_acc = fusion_results['Regional']['results_weighted']['accuracy_mean']
    for eeg_name in fusion_results.keys():
        if eeg_name != 'Regional':
            acc = fusion_results[eeg_name]['results_weighted']['accuracy_mean']
            diff = acc - baseline_acc
            print(f"   {eeg_name:20s}: {diff:+.4f} vs Regional")

print(f"\n4. RECOMMENDATIONS:")
if best_eeg_contrib['Weight'] > 2.0:
    print(f"   ✓ EEG shows meaningful contribution ({best_eeg_contrib['Weight']:.1f}%)")
    print(f"   → Use {best_eeg_contrib['EEG Features']} features for final analysis")
else:
    print(f"   • EEG contribution remains low across all feature sets (<2%)")
    print(f"   • Behavioral features dominate prediction")
    print(f"   → Consider: signal quality, time alignment, or other EEG features")

print(f"\n{'='*70}")

## 8. Save Results

In [ ]:
# Save all results
output_dir = f'../../data/results/fusion_models_eeg_{TIMEFRAME}'
Path(output_dir).mkdir(parents=True, exist_ok=True)

# 1. Overall comparison
save_results(comparison_df, f'{output_dir}/multimodal_eeg_comparison.csv')

# 2. Modality weights
save_results(weights_df, f'{output_dir}/multimodal_eeg_modality_weights.csv')

# 3. Detailed results for each EEG feature set
for eeg_name, result in fusion_results.items():
    safe_name = eeg_name.replace('+', '_').replace(' ', '_').lower()
    
    # Subject accuracies
    subject_df = pd.DataFrame({
        'subject_id': list(result['results_weighted']['subject_accs'].keys()),
        'accuracy': list(result['results_weighted']['subject_accs'].values())
    })
    save_results(subject_df, f'{output_dir}/multimodal_{safe_name}_subject_accuracies.csv')

print(f"\nAll results saved to: {output_dir}/")